In [ ]:
import numpy as np
import scipy as sp
import scipy.sparse as sparse
import scipy.sparse.linalg as sla
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def genA(n, dx):
    e = np.ones((n,))
    data = np.vstack((-e,2*e,-e))
    diags = np.array([-1, 0, 1])
    A = (1/dx**2) * sparse.spdiags(data, diags, n, n).tocsr()
    
    A[0,0] = 1
    A[0,1] = 0
    A[1,0] = 0
    A[-1,-1] = 1
    A[-1,-2] = 0
    A[-2,-1] = 0
    return A

def genuex(x, a, b):
    return np.sin(np.pi*(x-a)/b)

def genf(x, a, b):
    return (np.pi/b)**2 * np.sin(np.pi*(x-a)/b)

In [ ]:
#n1list = np.array([11,101,1001])
#n2list = n1list
n1list = np.array([8, 16, 32]) # grid 1 sizes
n2list = np.array([8, 20, 40]) # grid 2 sizes
plt.figure(figsize=(10,4))
print("%10s  %10s" %('grid-1 L2', 'grid-2 L2'))
print("%10s  %10s" %('---------', '---------'))

# grid 1: [a1,b1]
a1 = 0
b1 = 1
# grid 2: [a2,b2]
a2 = 0.8
b2 = 1.4





In [ ]:
# Additive Schwarz

e1s = []
e2s = []

for ni in range(len(n1list)):
    n1 = n1list[ni]
    n2 = n2list[ni]
    

    x1 = np.linspace(a1, b1, n1)
    dx1 = x1[1] - x1[0]
    x2 = np.linspace(a2, b2, n2)
    dx2 = x2[1] - x2[0]

    j1 = [np.where(x1<=a2)[0][-1], np.where(x1>=a2)[0][0]]
    j2 = [np.where(x2<=b1)[0][-1], np.where(x2>=b1)[0][0]]
   
    A1 = genA(n1, dx1)
    u1 = np.random.rand(n1)
    f1 = genf(x1, a1, b2)
    f1[0] = 0 # set bc

    A2 = genA(n2, dx2)
    u2 = np.random.rand(n2)
    f2 = genf(x2, a1, b2)
    f2[-1] = 0 # set bc

    for i in range(20):
        u1 = sla.spsolve(A1, f1)
        u2 = sla.spsolve(A2, f2) 
        
        # set the right hand side for the updated boundary condition
        f1 = genf(x1, a1, b2)
        f1[0] = 0 # set bc
        # now set other BC to the linear interpolant
        bc1 = ((u2[j2[1]] - u2[j2[0]])/dx2) * (x1[-1] - x2[j2[0]]) + u2[j2[0]]
        f1[-1] = bc1
        f1[-2] += bc1 / dx1**2
        
        # set the right hand side for the updated boundary condition
        f2 = genf(x2, a1, b2)
        f2[-1] = 0 # set bc
        # now set other BC to the linear interpolant
        bc2 = ((u1[j1[1]] - u1[j1[0]])/dx1) * (x2[0] - x1[j1[0]]) + u1[j1[0]] 
        f2[0] = bc2
        f2[1] += bc2 / dx2**2

        # compute some errors
        uex1 = genuex(x1,a1,b2)
        e1 = np.sqrt(dx1 * np.sum(np.abs(uex1 - u1)))
        uex2 = genuex(x2,a1,b2)
        e2 = np.sqrt(dx1 * np.sum(np.abs(uex2 - u2)))
        e1s.append(e1)
        e2s.append(e2)
        if i>0 and ni==0:
            plt.plot(x1, u1, 'b-', x2, u2, 'r-', x1, genuex(x1,a1,b2), 'b:', x2, genuex(x2,a1,b2), 'r:', lw=2)
            plt.xlabel(r'$x$')
            plt.ylabel(r'$\phi$')
        
    print("%10.2e  %10.2e" %(e1, e2))
    plt.axis('tight')
    plt.savefig('u.pdf')


In [ ]:
plt.plot(x1, u1, 'b-', x2, u2, 'r-', x1, genuex(x1,a1,b2), 'b:', x2, genuex(x2,a1,b2), 'r:')

In [ ]:
plt.figure(figsize=(4,4))
plt.semilogy(e1s, 'b-', label='Left Grid', lw=2)
plt.semilogy(e2s, 'r-', label='Right Grid', lw=2)
plt.legend(frameon=False)
plt.xlabel('iterations')
plt.ylabel('error')
plt.savefig('error.pdf')

In [ ]:
# Multiplicative Schwarz
e1s = []
e2s = []

for ni in range(len(n1list)):
    n1 = n1list[ni]
    n2 = n2list[ni]
    

    x1 = np.linspace(a1, b1, n1)
    dx1 = x1[1] - x1[0]
    x2 = np.linspace(a2, b2, n2)
    dx2 = x2[1] - x2[0]

    j1 = [np.where(x1<=a2)[0][-1], np.where(x1>=a2)[0][0]]
    j2 = [np.where(x2<=b1)[0][-1], np.where(x2>=b1)[0][0]]
   
    A1 = genA(n1, dx1)
    u1 = np.random.rand(n1)
    f1 = genf(x1, a1, b2)
    f1[0] = 0 # set bc

    A2 = genA(n2, dx2)
    u2 = np.zeros(n2)
    f2 = genf(x2, a1, b2)
    f2[-1] = 0 # set bc

    for i in range(20):
        
        # set the right hand side for the updated boundary condition
        f1 = genf(x1, a1, b2)
        f1[0] = 0 # set bc
        # now set other BC to the linear interpolant
        bc1 = ((u2[j2[1]] - u2[j2[0]])/dx2) * (x1[-1] - x2[j2[0]]) + u2[j2[0]]
        f1[-1] = bc1
        f1[-2] += bc1 / dx1**2
        
        # solve in the first domain
        u1 = sla.spsolve(A1, f1)
        
        
        # set the right hand side for the updated boundary condition
        f2 = genf(x2, a1, b2)
        f2[-1] = 0 # set bc
        # now set other BC to the linear interpolant
        bc2 = ((u1[j1[1]] - u1[j1[0]])/dx1) * (x2[0] - x1[j1[0]]) + u1[j1[0]] 
        f2[0] = bc2
        f2[1] += bc2 / dx2**2
        
        # solve in second domain
        u2 = sla.spsolve(A2, f2) 
        
        # compute some errors
        uex1 = genuex(x1,a1,b2)
        e1 = np.sqrt(dx1 * np.sum(np.abs(uex1 - u1)))
        uex2 = genuex(x2,a1,b2)
        e2 = np.sqrt(dx1 * np.sum(np.abs(uex2 - u2)))
        e1s.append(e1)
        e2s.append(e2)
        if i>0 and ni==0:
            plt.plot(x1, u1, 'b-', x2, u2, 'r-', x1, genuex(x1,a1,b2), 'b:', x2, genuex(x2,a1,b2), 'r:', lw=2)
            plt.xlabel(r'$x$')
            plt.ylabel(r'$\phi$')
        
    print("%10.2e  %10.2e" %(e1, e2))
    plt.axis('tight')
    plt.savefig('u.pdf')



In [ ]:
plt.plot(x1, u1, 'b-', x2, u2, 'r-', x1, genuex(x1,a1,b2), 'b:', x2, genuex(x2,a1,b2), 'r:')

In [ ]:
plt.figure(figsize=(4,4))
plt.semilogy(e1s, 'b-', label='Left Grid', lw=2)
plt.semilogy(e2s, 'r-', label='Right Grid', lw=2)
plt.legend(frameon=False)
plt.xlabel('iterations')
plt.ylabel('error')
plt.savefig('error.pdf')